In [1]:
import json
import os
from typing import Tuple

import numpy as np
import pandas as pd
import torch

from direction_learning import DirVectors
from logging_setup import create_logger
from phi_3_5_constants import dsets_index_path, token_lengths_path, train_split_records_path, \
    validation_split_records_path, probes_folder, dsets_folder, finalized_activations_dir, hidden_state_size, \
    directions_results_folder, misc_datasets_index_path, four_way_topics_index_path
from phi_3_5_probe import ProbesForDataset, train_probes_for_dset

In [2]:
logger = create_logger(__name__)

In [3]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")
num_dsets = dsets_index_df.shape[0]

In [4]:
with token_lengths_path.open("r") as f:
    record_lengths_in_tokens = json.load(f)
with train_split_records_path.open("r") as f:
    train_split_record_idxs = json.load(f)
with validation_split_records_path.open("r") as f:
    validation_split_record_idxs = json.load(f)

In [5]:
probes_folder.mkdir(exist_ok=True)

In [6]:
all_dsets_activations: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]
all_dsets_labels: list[torch.Tensor] = [torch.zeros(1) for _ in range(num_dsets)]


In [7]:
all_dsets_probes: dict[Tuple[int,...], ProbesForDataset] = {}

In [13]:
# top-level key is the name of a topic that has pos/neg/conj/disj variants, second level key is one of those variant names
with four_way_topics_index_path.open("r") as f:
    dset_idxs_for_4way_topics: dict[str, dict[str, int]] = json.load(f)
with misc_datasets_index_path.open("r") as f:
    idxs_for_other_dsets: dict[str, int] = json.load(f)

In [14]:
for dset_idx, dset_dtls in dsets_index_df.iterrows():
    categ_nm = dset_dtls["Categ_Folder"]
    dset_file_nm = dset_dtls["Dataset_File"]
    dset_nm = os.path.splitext(dset_file_nm)[0]
    
    dataset = pd.read_csv(dsets_folder / categ_nm / dset_file_nm)
    dset_size = dataset.shape[0]
    dset_labels = torch.from_numpy(dataset['label'].to_numpy().astype(np.float32)[:, np.newaxis])
    all_dsets_labels[dset_idx] = dset_labels
    
    activs_path = finalized_activations_dir / categ_nm / (dset_nm + ".pt")
    relevant_activations = torch.load(activs_path, weights_only=True)
    assert relevant_activations.shape == (2, dset_size, hidden_state_size)
    all_dsets_activations[dset_idx] = relevant_activations
    
    train_split_activs = relevant_activations[:, train_split_record_idxs[str(dset_idx)], :]
    train_split_truth_labels = dset_labels[train_split_record_idxs[str(dset_idx)], :]
    
    val_split_activs = relevant_activations[:, validation_split_record_idxs[str(dset_idx)], :]
    val_split_truth_labels = dset_labels[validation_split_record_idxs[str(dset_idx)], :]
    
    dirs_for_dset_path = directions_results_folder / categ_nm / (dset_nm + ".pt")
    if not dirs_for_dset_path.exists():
        logger.warning(f"Directions for {dset_nm} not found, stopping (presuming that we've reached the end of the set of datasets whose directions have been computed)")
        break
    dirs_dict = torch.load(dirs_for_dset_path, weights_only=True)
    assert isinstance(dirs_dict, dict)
    mean_truth_polarity_dirs_for_dset = DirVectors(**dirs_dict)
    
    probes = train_probes_for_dset(probes_folder / categ_nm, dset_nm, train_split_activs, train_split_truth_labels,
                          val_split_activs, val_split_truth_labels, mean_truth_polarity_dirs_for_dset)
    all_dsets_probes[(dset_idx,)] = probes

2024-11-09 00:04:40,543;phi_3_5_probe;INFO:training the layer18 probe for 131 records of data animal_class in the location trained_probes\animal_class
2024-11-09 00:04:59,184;phi_3_5_probe;INFO:scaling learning rate up to 1.000000e-04 at epoch 3072 because loss (avg'd over 10 timesteps) has improved by 4.015527e-04 since 1024 epochs ago and last 1024 epochs have included a minimal number of stagnant or backsliding epochs


KeyboardInterrupt: 

In [ ]:
for topic_nm, variants_dset_idxs in dset_idxs_for_4way_topics.items():
    affirm_idx = variants_dset_idxs["affirm"]
    affirm_train_activs = all_dsets_activations[affirm_idx][:, train_split_record_idxs[str(affirm_idx)], :]
    affirm_train_truth_labels = all_dsets_labels[affirm_idx][train_split_record_idxs[str(affirm_idx)], :]
    affirm_validation_activs = all_dsets_activations[affirm_idx][:, validation_split_record_idxs[str(affirm_idx)], :]
    affirm_validation_truth_labels = all_dsets_labels[affirm_idx][validation_split_record_idxs[str(affirm_idx)], :]
    
    neg_idx = variants_dset_idxs["neg"]
    neg_train_activs = all_dsets_activations[neg_idx][:, train_split_record_idxs[str(neg_idx)], :]
    neg_train_truth_labels = all_dsets_labels[neg_idx][train_split_record_idxs[str(neg_idx)], :]
    neg_validation_activs = all_dsets_activations[neg_idx][:, validation_split_record_idxs[str(neg_idx)], :]
    neg_validation_truth_labels = all_dsets_labels[neg_idx][validation_split_record_idxs[str(neg_idx)], :]
    
    conj_idx = variants_dset_idxs["conj"]
    conj_train_activs = all_dsets_activations[conj_idx][:, train_split_record_idxs[str(conj_idx)], :]
    conj_train_truth_labels = all_dsets_labels[conj_idx][train_split_record_idxs[str(conj_idx)], :]
    conj_validation_activs = all_dsets_activations[conj_idx][:, validation_split_record_idxs[str(conj_idx)], :]
    conj_validation_truth_labels = all_dsets_labels[conj_idx][validation_split_record_idxs[str(conj_idx)], :]
    
    disj_idx = variants_dset_idxs["disj"]
    disj_train_activs = all_dsets_activations[disj_idx][:, train_split_record_idxs[str(disj_idx)], :]
    disj_train_truth_labels = all_dsets_labels[disj_idx][train_split_record_idxs[str(disj_idx)], :]
    disj_validation_activs = all_dsets_activations[disj_idx][:, validation_split_record_idxs[str(disj_idx)], :]
    disj_validation_truth_labels = all_dsets_labels[disj_idx][validation_split_record_idxs[str(disj_idx)], :]
    
    affirm_neg_key = tuple(sorted([affirm_idx, neg_idx]))
    dirs_for_affirm_neg_path = directions_results_folder / topic_nm / "affirm_neg.pt"
    if not dirs_for_affirm_neg_path.exists():
        logger.warning(f"Directions for {topic_nm} affirm_neg not found, stopping (presuming that we've reached the end of the set of datasets whose directions have been computed)")
        break
    affirm_neg_dirs_dict = torch.load(dirs_for_affirm_neg_path, weights_only=True)
    assert isinstance(affirm_neg_dirs_dict, dict)
    affirm_neg_dirs = DirVectors(**affirm_neg_dirs_dict)
    affirm_neg_probes = train_probes_for_dset(probes_folder / topic_nm, "affirm_neg", torch.concat((affirm_train_activs, neg_train_activs), dim=1), torch.concat((affirm_train_truth_labels, neg_train_truth_labels), dim=0), torch.concat((affirm_validation_activs, neg_validation_activs), dim=1), torch.concat((affirm_validation_truth_labels, neg_validation_truth_labels), dim=0), affirm_neg_dirs)
    all_dsets_probes[affirm_neg_key] = affirm_neg_probes
    
    affirm_disj_key = tuple(sorted([affirm_idx, disj_idx]))
    dirs_for_affirm_disj_path = directions_results_folder / topic_nm / "affirm_disj.pt"
    if not dirs_for_affirm_disj_path.exists():
        logger.warning(f"Directions for {topic_nm} affirm_disj not found, stopping (presuming that we've reached the end of the set of datasets whose directions have been computed)")
        break
    affirm_disj_dirs_dict = torch.load(dirs_for_affirm_disj_path, weights_only=True)
    assert isinstance(affirm_disj_dirs_dict, dict)
    affirm_disj_dirs = DirVectors(**affirm_disj_dirs_dict)
    affirm_disj_probes = train_probes_for_dset(probes_folder / topic_nm, "affirm_disj", torch.concat((affirm_train_activs, disj_train_activs), dim=1), torch.concat((affirm_train_truth_labels, disj_train_truth_labels), dim=0), torch.concat((affirm_validation_activs, disj_validation_activs), dim=1), torch.concat((affirm_validation_truth_labels, disj_validation_truth_labels), dim=0), affirm_disj_dirs)
    all_dsets_probes[affirm_disj_key] = affirm_disj_probes
    
    neg_conj_key = tuple(sorted([neg_idx, conj_idx]))
    dirs_for_neg_conj_path = directions_results_folder / topic_nm / "neg_conj.pt"
    if not dirs_for_neg_conj_path.exists():
        logger.warning(f"Directions for {topic_nm} neg_conj not found, stopping (presuming that we've reached the end of the set of datasets whose directions have been computed)")
        break
    neg_conj_dirs_dict = torch.load(dirs_for_neg_conj_path, weights_only=True)
    assert isinstance(neg_conj_dirs_dict, dict)
    neg_conj_dirs = DirVectors(**neg_conj_dirs_dict)
    neg_conj_probes = train_probes_for_dset(probes_folder / topic_nm, "neg_conj", torch.concat((neg_train_activs, conj_train_activs), dim=1), torch.concat((neg_train_truth_labels, conj_train_truth_labels), dim=0), torch.concat((neg_validation_activs, conj_validation_activs), dim=1), torch.concat((neg_validation_truth_labels, conj_validation_truth_labels), dim=0), neg_conj_dirs)
    all_dsets_probes[neg_conj_key] = neg_conj_probes
    
    affirm_neg_conj_disj_key = tuple(sorted([affirm_idx, neg_idx, conj_idx, disj_idx]))
    dirs_for_affirm_neg_conj_disj_path = directions_results_folder / topic_nm / "affirm_neg_conj_disj.pt"
    if not dirs_for_affirm_neg_conj_disj_path.exists():
        logger.warning(f"Directions for {topic_nm} affirm_neg_conj_disj not found, stopping (presuming that we've reached the end of the set of datasets whose directions have been computed)")
        break
    affirm_neg_conj_disj_dirs_dict = torch.load(dirs_for_affirm_neg_conj_disj_path, weights_only=True)
    assert isinstance(affirm_neg_conj_disj_dirs_dict, dict)
    affirm_neg_conj_disj_dirs = DirVectors(**affirm_neg_conj_disj_dirs_dict)
    affirm_neg_conj_disj_probes = train_probes_for_dset(probes_folder / topic_nm, "affirm_neg_conj_disj", torch.concat((affirm_train_activs, neg_train_activs, conj_train_activs, disj_train_activs), dim=1), torch.concat((affirm_train_truth_labels, neg_train_truth_labels, conj_train_truth_labels, disj_train_truth_labels), dim=0), torch.concat((affirm_validation_activs, neg_validation_activs, conj_validation_activs, disj_validation_activs), dim=1), torch.concat((affirm_validation_truth_labels, neg_validation_truth_labels, conj_validation_truth_labels, disj_validation_truth_labels), dim=0), affirm_neg_conj_disj_dirs)
    all_dsets_probes[affirm_neg_conj_disj_key] = affirm_neg_conj_disj_probes


In [ ]:
unambig_lie_idx = idxs_for_other_dsets["unambiguous_lie"]
unambig_lie_train_activs = all_dsets_activations[unambig_lie_idx][:, train_split_record_idxs[str(unambig_lie_idx)], :]
unambig_lie_train_truth_labels = all_dsets_labels[unambig_lie_idx][train_split_record_idxs[str(unambig_lie_idx)], :]
unambig_lie_validation_activs = all_dsets_activations[unambig_lie_idx][:, validation_split_record_idxs[str(unambig_lie_idx)], :]
unambig_lie_validation_truth_labels = all_dsets_labels[unambig_lie_idx][validation_split_record_idxs[str(unambig_lie_idx)], :]

unambig_truth_idx = idxs_for_other_dsets["unambiguous_truthful_reply"]
unambig_truth_train_activs = all_dsets_activations[unambig_truth_idx][:, train_split_record_idxs[str(unambig_truth_idx)], :]
unambig_truth_train_truth_labels = all_dsets_labels[unambig_truth_idx][train_split_record_idxs[str(unambig_truth_idx)], :]
unambig_truth_validation_activs = all_dsets_activations[unambig_truth_idx][:, validation_split_record_idxs[str(unambig_truth_idx)], :]
unambig_truth_validation_truth_labels = all_dsets_labels[unambig_truth_idx][validation_split_record_idxs[str(unambig_truth_idx)], :]

ambig_truth_idx = idxs_for_other_dsets["ambiguous_truthful_reply"]
ambig_truth_train_activs = all_dsets_activations[ambig_truth_idx][:, train_split_record_idxs[str(ambig_truth_idx)], :]
ambig_truth_train_truth_labels = all_dsets_labels[ambig_truth_idx][train_split_record_idxs[str(ambig_truth_idx)], :]
ambig_truth_validation_activs = all_dsets_activations[ambig_truth_idx][:, validation_split_record_idxs[str(ambig_truth_idx)], :]
ambig_truth_validation_truth_labels = all_dsets_labels[ambig_truth_idx][validation_split_record_idxs[str(ambig_truth_idx)], :]

selected_real_world_train_activs = torch.concat(
    (unambig_lie_train_activs, unambig_truth_train_activs, ambig_truth_train_activs), dim=1)
selected_real_world_train_truth_labels = torch.concat(
    (unambig_lie_train_truth_labels, unambig_truth_train_truth_labels, ambig_truth_train_truth_labels), dim=0)
selected_real_world_validation_activs = torch.concat(
    (unambig_lie_validation_activs, unambig_truth_validation_activs, ambig_truth_validation_activs), dim=1)
selected_real_world_validation_truth_labels = torch.concat(
    (unambig_lie_validation_truth_labels, unambig_truth_validation_truth_labels, ambig_truth_validation_truth_labels), dim=0)

real_world_multi_dset_idxs = [unambig_lie_idx, unambig_truth_idx, ambig_truth_idx]
real_world_multi_dset_scenario_key = tuple(sorted(real_world_multi_dset_idxs))
dirs_for_selected_real_world_path = directions_results_folder / "real_world_scenarios" / "unambig_lie_unambig_truth_ambig_truth.pt"
if not dirs_for_selected_real_world_path.exists():
    logger.warning(f"Directions for unambig_lie_unambig_truth_ambig_truth not found")
else:
    selected_real_world_dirs_dict = torch.load(dirs_for_selected_real_world_path, weights_only=True)
    assert isinstance(selected_real_world_dirs_dict, dict)
    selected_real_world_dirs = DirVectors(**selected_real_world_dirs_dict)
    selected_real_world_probes = train_probes_for_dset(probes_folder / "real_world_scenarios", "unambig_lie_unambig_truth_ambig_truth", selected_real_world_train_activs, selected_real_world_train_truth_labels, selected_real_world_validation_activs, selected_real_world_validation_truth_labels, selected_real_world_dirs)
    all_dsets_probes[real_world_multi_dset_scenario_key] = selected_real_world_probes


In [ ]:
multi_topics_dset_idxs = []
multi_topics_affirm_neg_conj_train_activs = torch.zeros((2, 1, hidden_state_size))
multi_topics_affirm_neg_conj_train_truth_labels = torch.zeros((1, 1))
multi_topics_affirm_neg_conj_validation_activs = torch.zeros((2, 1, hidden_state_size))
multi_topics_affirm_neg_conj_validation_truth_labels = torch.zeros((1, 1))

various_categories_dset_idxs = []
various_categories_train_activs = torch.zeros((2, 1, hidden_state_size))
various_categories_train_truth_labels = torch.zeros((1, 1))
various_categories_validation_activs = torch.zeros((2, 1, hidden_state_size))
various_categories_validation_truth_labels = torch.zeros((1, 1))

for topic_nm, variants_dset_idxs in dset_idxs_for_4way_topics.items():
    if topic_nm not in ("animal_class", "element_symbols", "facts", "inventors"):
        continue
    logger.info(f"concatenating data for the topic {topic_nm} to prepare for multi-topic/category experiments")
    affirm_idx = variants_dset_idxs["affirm"]
    affirm_train_activs = all_dsets_activations[affirm_idx][:, train_split_record_idxs[str(affirm_idx)], :]
    affirm_train_truth_labels = all_dsets_labels[affirm_idx][train_split_record_idxs[str(affirm_idx)], :]
    affirm_validation_activs = all_dsets_activations[affirm_idx][:, validation_split_record_idxs[str(affirm_idx)], :]
    affirm_validation_truth_labels = all_dsets_labels[affirm_idx][validation_split_record_idxs[str(affirm_idx)], :]

    neg_idx = variants_dset_idxs["neg"]
    neg_train_activs = all_dsets_activations[neg_idx][:, train_split_record_idxs[str(neg_idx)], :]
    neg_train_truth_labels = all_dsets_labels[neg_idx][train_split_record_idxs[str(neg_idx)], :]
    neg_validation_activs = all_dsets_activations[neg_idx][:, validation_split_record_idxs[str(neg_idx)], :]
    neg_validation_truth_labels = all_dsets_labels[neg_idx][validation_split_record_idxs[str(neg_idx)], :]

    conj_idx = variants_dset_idxs["conj"]
    conj_train_activs = all_dsets_activations[conj_idx][:, train_split_record_idxs[str(conj_idx)], :]
    conj_train_truth_labels = all_dsets_labels[conj_idx][train_split_record_idxs[str(conj_idx)], :]
    conj_validation_activs = all_dsets_activations[conj_idx][:, validation_split_record_idxs[str(conj_idx)], :]
    conj_validation_truth_labels = all_dsets_labels[conj_idx][validation_split_record_idxs[str(conj_idx)], :]

    multi_topics_affirm_neg_conj_train_activs = torch.concat((
        multi_topics_affirm_neg_conj_train_activs, affirm_train_activs, neg_train_activs, conj_train_activs), dim=1)
    multi_topics_affirm_neg_conj_train_truth_labels = torch.concat((
        multi_topics_affirm_neg_conj_train_truth_labels, affirm_train_truth_labels, neg_train_truth_labels,
        conj_train_truth_labels), dim=0)
    multi_topics_affirm_neg_conj_validation_activs = torch.concat((
        multi_topics_affirm_neg_conj_validation_activs, affirm_validation_activs, neg_validation_activs, conj_validation_activs), dim=1)
    multi_topics_affirm_neg_conj_validation_truth_labels = torch.concat((
        multi_topics_affirm_neg_conj_validation_truth_labels, affirm_validation_truth_labels, neg_validation_truth_labels,
        conj_validation_truth_labels), dim=0)
    multi_topics_dset_idxs.extend([affirm_idx, neg_idx, conj_idx])

    various_categories_train_activs = torch.concat((
        various_categories_train_activs, affirm_train_activs, neg_train_activs), dim=1)
    various_categories_train_truth_labels = torch.concat((
        various_categories_train_truth_labels, affirm_train_truth_labels, neg_train_truth_labels), dim=0)
    various_categories_validation_activs = torch.concat((
        various_categories_validation_activs, affirm_validation_activs, neg_validation_activs), dim=1)
    various_categories_validation_truth_labels = torch.concat((
        various_categories_validation_truth_labels, affirm_validation_truth_labels, neg_validation_truth_labels), dim=0)
    various_categories_dset_idxs.extend([affirm_idx, neg_idx])

multi_topics_affirm_neg_conj_train_activs = multi_topics_affirm_neg_conj_train_activs[:, 1:, :]
multi_topics_affirm_neg_conj_train_truth_labels = multi_topics_affirm_neg_conj_train_truth_labels[1:, :]
multi_topics_affirm_neg_conj_validation_activs = multi_topics_affirm_neg_conj_validation_activs[:, 1:, :]
multi_topics_affirm_neg_conj_validation_truth_labels = multi_topics_affirm_neg_conj_validation_truth_labels[1:, :]

various_categories_train_activs = various_categories_train_activs[:, 1:, :]
various_categories_train_truth_labels = various_categories_train_truth_labels[1:, :]
various_categories_validation_activs = various_categories_validation_activs[:, 1:, :]
various_categories_validation_truth_labels = various_categories_validation_truth_labels[1:, :]

In [ ]:
multi_topics_affirm_neg_conj_key = tuple(sorted(multi_topics_dset_idxs))

dirs_for_multi_topics_affirm_neg_conj_path = directions_results_folder / "multi_topics_affirm_neg_conj.pt"
if not dirs_for_multi_topics_affirm_neg_conj_path.exists():
    logger.warning(f"Directions for multi_topics_affirm_neg_conj not found")
else:
    multi_topics_affirm_neg_conj_dirs_dict = torch.load(dirs_for_multi_topics_affirm_neg_conj_path, weights_only=True)
    assert isinstance(multi_topics_affirm_neg_conj_dirs_dict, dict)
    multi_topics_affirm_neg_conj_dirs = DirVectors(**multi_topics_affirm_neg_conj_dirs_dict)
    multi_topics_affirm_neg_conj_probes = train_probes_for_dset(probes_folder, "multi_topics_affirm_neg_conj", multi_topics_affirm_neg_conj_train_activs, multi_topics_affirm_neg_conj_train_truth_labels, multi_topics_affirm_neg_conj_validation_activs, multi_topics_affirm_neg_conj_validation_truth_labels, multi_topics_affirm_neg_conj_dirs)
    all_dsets_probes[multi_topics_affirm_neg_conj_key] = multi_topics_affirm_neg_conj_probes

In [ ]:

smaller_than_idx = idxs_for_other_dsets["smaller_than"]
smaller_than_train_activs = all_dsets_activations[smaller_than_idx][:, train_split_record_idxs[str(smaller_than_idx)], :]
smaller_than_train_truth_labels = all_dsets_labels[smaller_than_idx][train_split_record_idxs[str(smaller_than_idx)], :]
smaller_than_validation_activs = all_dsets_activations[smaller_than_idx][:, validation_split_record_idxs[str(smaller_than_idx)], :]
smaller_than_validation_truth_labels = all_dsets_labels[smaller_than_idx][validation_split_record_idxs[str(smaller_than_idx)], :]

common_claim_idx = idxs_for_other_dsets["common_claim_true_false"]
common_claim_train_activs = all_dsets_activations[common_claim_idx][:, train_split_record_idxs[str(common_claim_idx)], :]
common_claim_train_truth_labels = all_dsets_labels[common_claim_idx][train_split_record_idxs[str(common_claim_idx)], :]
common_claim_validation_activs = all_dsets_activations[common_claim_idx][:, validation_split_record_idxs[str(common_claim_idx)], :]
common_claim_validation_truth_labels = all_dsets_labels[common_claim_idx][validation_split_record_idxs[str(common_claim_idx)], :]

various_categories_train_activs = torch.concat((
    various_categories_train_activs, selected_real_world_train_activs, smaller_than_train_activs, 
    common_claim_train_activs), dim=1)
various_categories_train_truth_labels = torch.concat((
    various_categories_train_truth_labels, selected_real_world_train_truth_labels, smaller_than_train_truth_labels,
    common_claim_train_truth_labels), dim=0)
various_categories_validation_activs = torch.concat((
    various_categories_validation_activs, selected_real_world_validation_activs, smaller_than_validation_activs, 
    common_claim_validation_activs), dim=1)
various_categories_validation_truth_labels = torch.concat((
    various_categories_validation_truth_labels, selected_real_world_validation_truth_labels, smaller_than_validation_truth_labels,
    common_claim_validation_truth_labels), dim=0)
various_categories_dset_idxs.extend(real_world_multi_dset_idxs)
various_categories_dset_idxs.extend([smaller_than_idx, common_claim_idx])
various_categories_scenario_key = tuple(sorted(various_categories_dset_idxs))

dirs_for_various_categories_path = directions_results_folder / "various_categories.pt"
if not dirs_for_various_categories_path.exists():
    logger.warning(f"Directions for various_categories not found")
else:
    various_categories_dirs_dict = torch.load(dirs_for_various_categories_path, weights_only=True)
    assert isinstance(various_categories_dirs_dict, dict)
    various_categories_dirs = DirVectors(**various_categories_dirs_dict)
    various_categories_probes = train_probes_for_dset(probes_folder, "various_categories", various_categories_train_activs, various_categories_train_truth_labels, various_categories_validation_activs, various_categories_validation_truth_labels, various_categories_dirs)
    all_dsets_probes[various_categories_scenario_key] = various_categories_probes


In [ ]:
# TODO check for relationship between final 'cost' of OLS process for a given dataset-scenario and layer choice (after normalizing for number of records) and the best validation loss that can be achieved by a probe using the given directions on the given dataset-scenario